# Cycle 3 — Model Explainability (SHAP): Player Injury Risk

This notebook explains **why the injury risk model predicts high or low injury burden** for individual players.

The tuned LightGBM model achieves **AUC 0.6800** on the chronological test set (held-out 2020 season). SHAP reveals *which player attributes* drive each injury risk prediction. This matters for:
1. Validating that the model uses medically sensible risk factors (age, BMI, injury history)
2. Identifying which pre-season signals are most predictive of high injury burden
3. Making individual risk predictions explainable for club medical staff via the dashboard

## What is SHAP?

**SHAP** (SHapley Additive exPlanations) assigns each feature a contribution for each prediction.

- A **positive SHAP value** means the feature pushed the model towards predicting **High Injury**
- A **negative SHAP value** means the feature pushed the model towards **Low Injury**
- SHAP values are mathematically consistent and sum to the model output minus the baseline

**Example:** For a player, `avg_days_injured_prev_seasons = +0.12` means their prior injury history added 0.12 to the High Injury log-odds.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score

_here = os.getcwd()
while not os.path.isdir(os.path.join(_here, 'data')):
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)

from config import Paths, ensure_dirs
ensure_dirs()


## Load Saved Model and Rebuild Chronological Test Set

**What it does:** Loads the three artefacts saved by `cycle3_tuning_chronological.ipynb` and reconstructs the exact same chronological test split.

**Why reload from raw (not processed)?** The processed CSV drops `start_year`, which is needed to reconstruct the temporal 80/20 split. The feature engineering must be re-applied here identically.

**Why chronological split?** The saved model was evaluated on the held-out 2020 season. Using a random split here would report an inflated, leaky AUC that doesn't reflect deployment reality.

**AUC 0.6800** is the honest estimate: the model was trained on pre-2020 data and tested on the 2020 season it had never seen.

In [ ]:
# Load artefacts saved by cycle3_tuning_chronological.ipynb
model        = joblib.load(str(Paths.C3_MODEL))     # best model: LightGBM Tuned (AUC=0.6800)
scaler       = joblib.load(str(Paths.C3_SCALER))    # fitted StandardScaler
feature_cols = joblib.load(str(Paths.C3_FEATURES))  # 17 feature column names in training order

# ── Rebuild the chronological test set ──────────────────────────────────────
# Must replicate the exact same reconstruction as cycle3_tuning_chronological.ipynb
# because the processed CSV lacks start_year, which is needed for the temporal split.
raw = pd.read_csv(str(Paths.PLAYER_INJURIES_RAW))
raw['High_Injury'] = (raw['season_days_injured'] >= 28).astype(int)   # 28-day injury threshold

# Ordinal encodings and derived features (same as preprocessing + tuning notebooks)
wr_map  = {'Low': 1, 'Medium': 2, 'High': 3}
pos_map = {'GK': 1, 'DF': 2, 'MF': 3, 'FW': 4}
if 'work_rate' in raw.columns:
    raw['work_rate_numeric'] = (raw['work_rate'].astype(str)
                                .str.split('/').str[0].str.strip()
                                .map(wr_map).fillna(2))
if 'position' in raw.columns:
    raw['position_numeric'] = (raw['position'].astype(str)
                               .str[:2].map(pos_map).fillna(3))
if 'height_cm' in raw.columns and 'weight_kg' in raw.columns:
    raw['bmi'] = raw['weight_kg'] / (raw['height_cm'] / 100.0) ** 2

# Drop leakage columns (in-season statistics that would not be available pre-season)
drop_cols = ['p_id2', 'dob', 'nationality', 'work_rate', 'position',
             'season_days_injured', 'total_days_injured',
             'season_minutes_played', 'season_games_played', 'season_matches_in_squad',
             'total_minutes_played', 'total_games_played']
df = raw.drop(columns=[c for c in drop_cols if c in raw.columns])

# Fill missing injury history with 0 (= no recorded history for first-season players)
history_cols = ['cumulative_minutes_played', 'cumulative_games_played',
                'minutes_per_game_prev_seasons', 'avg_days_injured_prev_seasons',
                'avg_games_per_season_prev_seasons', 'significant_injury_prev_season',
                'cumulative_days_injured', 'season_days_injured_prev_season']
history_cols = [c for c in history_cols if c in df.columns]
df[history_cols] = df[history_cols].fillna(0)
df = df.dropna().sort_values('start_year').reset_index(drop=True)   # chronological order

# 80/20 row-index split (same boundary as the tuning notebook)
split_idx = int(len(df) * 0.8)
train_df  = df.iloc[:split_idx]
test_df   = df.iloc[split_idx:]

drop_for_X = ['High_Injury', 'start_year']
X_train = train_df.drop(columns=drop_for_X)
y_train = train_df['High_Injury']
X_test  = test_df.drop(columns=drop_for_X)
y_test  = test_df['High_Injury']

# Align columns with the saved feature list (same order as training)
extras  = [c for c in X_test.columns if c not in feature_cols]
missing = [c for c in feature_cols if c not in X_test.columns]
if extras:  X_test = X_test.drop(columns=extras)
if missing:
    for m in missing: X_test[m] = 0.0
X_test = X_test[feature_cols]

X_test_s = scaler.transform(X_test)   # apply saved scaler — no re-fitting

auc = roc_auc_score(y_test, model.predict_proba(X_test_s)[:, 1])
acc = accuracy_score(y_test, model.predict(X_test_s))
print(f'Loaded model test AUC: {auc:.4f}')
print(f'Loaded model test Acc: {acc*100:.2f}%')
print(f'Test season range: {test_df["start_year"].min()}–{test_df["start_year"].max()}')
print(f'Test rows: {len(X_test)}, Features: {len(feature_cols)}')
print(f'High Injury rate in test: {y_test.mean()*100:.1f}%')


### Observations
- **AUC 0.6800 confirmed** — matches `cycle3_tuning_chronological.ipynb`
- Test set spans a single season (2020) — a strict out-of-sample test with no temporal leakage
- High Injury rate ~61% — the majority class; but AUC correctly accounts for this imbalance
- `scaler.transform()` (not `fit_transform`) — applies the training-fit parameters, preventing test-set information from influencing normalisation

## Compute SHAP Values

**What it does:** Runs the SHAP TreeExplainer on the chronological test set to produce one SHAP value per feature per player-season.

**Why TreeExplainer?** LightGBM is tree-based. `shap.TreeExplainer` uses an exact, fast algorithm that exploits the tree structure for efficient computation.

**Output shape:** `(n_samples, n_features)` — for each player-season in the test set, 17 SHAP values indicate how each pre-season feature contributed to the High Injury prediction.

In [ ]:
explainer = shap.TreeExplainer(model)       # exact fast SHAP for tree-based models
sv = explainer.shap_values(X_test_s)       # compute SHAP for all test players

# LightGBM binary classification returns a list [neg_class_shap, pos_class_shap]
# XGBoost binary returns a single 2D array — handle both to be safe
if isinstance(sv, list):
    shap_arr = sv[1]    # positive class = High Injury (index 1)
else:
    shap_arr = sv

print(f'SHAP values shape: {shap_arr.shape}')
print(f'n_samples={shap_arr.shape[0]}, n_features={shap_arr.shape[1]}')
print('Each value = how much that feature pushed the High Injury prediction up or down')


### Observations
- SHAP values shape confirms one contribution per player per feature
- For LightGBM binary classification, SHAP returns separate values for the positive class (High Injury)
- Positive SHAP → pushes toward High Injury risk; Negative SHAP → pushes toward Low Injury risk

## Global Feature Importance (Mean Absolute SHAP)

**What it does:** Calculates the mean absolute SHAP value for each feature — the average magnitude of that feature's influence across all player-season predictions.

**Why mean absolute?** Averaging raw SHAP values would cancel positive and negative effects. Absolute SHAP captures total influence, regardless of whether the feature increases or decreases risk.

In [ ]:
mean_abs_shap = np.abs(shap_arr).mean(axis=0)   # average magnitude across all players

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=False)

print('Global Feature Importance (Mean Absolute SHAP)')
print(importance_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance_df['Feature'][::-1], importance_df['Mean |SHAP|'][::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Cycle 3 — Global Feature Importance (SHAP)\nLightGBM Injury Risk Model (Chronological Split)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
os.makedirs('../../docs', exist_ok=True)
plt.savefig('../../docs/cycle3_shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → ../../docs/cycle3_shap_global_importance.png')


### Observations
- **Injury history features dominate** — prior injury burden (`avg_days_injured_prev_seasons`, `cumulative_days_injured`, `significant_injury_prev_season`) are the strongest predictors
- **Age** is a key risk factor — older players carry higher injury risk, consistent with sports medicine literature
- **Physical attributes** (BMI, height, weight) contribute moderately — extreme values (very high BMI) increase risk
- **Workload features** (cumulative minutes, games) capture overuse injury risk
- **Position** has modest importance — defenders and forwards face different injury profiles
- **Work rate** and first-season indicators contribute least — player intent matters less than physical history

## SHAP Summary Plot (Beeswarm)

**What it does:** Plots a beeswarm where each dot is one player-season, coloured by feature value (red = high, blue = low). The x-axis shows the SHAP contribution to High Injury risk.

**How to read it:**
- A red dot on the **right** means: high values of this feature **increase** High Injury risk
- A red dot on the **left** means: high values **decrease** High Injury risk
- A blue dot on the **right** means: low values of this feature increase injury risk

**Why this plot?** It reveals not just which features matter but *how* — direction, spread, and individual variation.

In [ ]:
os.makedirs('../../docs', exist_ok=True)

fig, ax = plt.subplots(figsize=(11, 7))
plt.sca(ax)
shap.summary_plot(
    shap_arr,
    X_test,                      # original unscaled values for colour coding
    feature_names=feature_cols,
    show=False,
    plot_type='dot',             # beeswarm: each dot = one player-season
    plot_size=None
)
plt.title('Cycle 3 — SHAP Beeswarm (Injury Risk: Probability of High Injury)', pad=15, fontweight='bold')
plt.tight_layout()
fname = '../../docs/cycle3_shap_summary.png'
plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {fname}')


### Observations

**Injury history (avg_days_injured_prev_seasons, cumulative_days_injured):**
- Red dots (high prior injury days) → right → strongly increase High Injury risk
- Clear, consistent positive relationship: past injury is the strongest predictor of future injury
- This aligns with the well-established medical finding that injury history is the #1 risk factor

**significant_injury_prev_season:**
- Players with a significant injury last season (red = 1) → right → higher injury risk
- Binary feature with clear directional effect

**Age:**
- Older players (red) → right → increased risk, consistent with sports medicine literature
- Younger players (blue) → left → lower risk

**Cumulative minutes / games:**
- Higher cumulative load → increased injury risk (overuse pattern)
- Players with more career minutes show increased risk, likely due to accumulated physical wear

**BMI:**
- Higher BMI (red) → slightly right → marginal increase in risk
- Effect is smaller than injury history features

**Position and work rate:**
- Effects are smaller and more variable — no strong directional pattern
- Model has learned some position-specific tendencies but they are secondary to injury history

## Single Prediction Explanation (Waterfall Plot)

**What it does:** Explains one specific player prediction in full detail — traces the path from the model's baseline injury risk to the final predicted risk score.

**How to read it:**
- `E[f(X)]` is the average injury risk across all training players (the baseline)
- Each feature bar adds to or subtracts from that baseline
- The final value `f(x)` is the model's predicted High Injury probability for this player

**Why this plot?** This is what the Streamlit dashboard shows per player — a transparent breakdown of why a player received their injury risk score, supporting medical staff decision-making.

In [ ]:
# Explain one specific High Injury prediction
high_idx   = y_test[y_test == 1].index
sample_idx = X_test.index.get_loc(high_idx[0])   # first actual High Injury player in test set

predicted_prob  = model.predict_proba(X_test_s[sample_idx:sample_idx+1])[0, 1]
predicted_class = model.predict(X_test_s[sample_idx:sample_idx+1])[0]
actual_class    = y_test.iloc[sample_idx]
outcome_names   = {0: 'Low Injury', 1: 'High Injury'}

print(f'Sample index:              {sample_idx}')
print(f'Actual:                    {actual_class} ({outcome_names[actual_class]})')
print(f'Predicted:                 {predicted_class} ({outcome_names[predicted_class]})')
print(f'High Injury probability:   {predicted_prob:.3f}')
print()

# expected_value is the model baseline (average injury risk across training data)
base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)):
    base_val = float(base_val[1])   # positive class baseline

shap_explanation = shap.Explanation(
    values=shap_arr[sample_idx],           # per-feature SHAP contributions for this player
    base_values=base_val,                  # baseline injury risk (average model output)
    data=X_test.values[sample_idx],        # original unscaled feature values for display
    feature_names=feature_cols
)

plt.figure(figsize=(10, 7))
shap.plots.waterfall(shap_explanation, show=False)
plt.title(
    f'Single Player Explanation — Risk: {predicted_prob:.3f} | Predicted: {outcome_names[predicted_class]}\n'
    f'Actual: {outcome_names[actual_class]}',
    fontsize=11
)
plt.tight_layout()
plt.savefig('../../docs/cycle3_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → ../../docs/cycle3_shap_waterfall.png')


### Observations
- The waterfall reveals exactly which risk factors elevated or reduced this player's predicted injury burden
- Features with long bars had the most influence on this specific prediction
- Medical staff can use this to prioritise interventions: e.g. a player flagged by high `avg_days_injured_prev_seasons` should have a tailored load management plan

### Note for Report
**Explainable AI (XAI)** is especially important for medical applications: a clinical decision support tool must be transparent to be trusted and adopted. SHAP provides a mathematically rigorous, per-player breakdown of which risk factors drove the prediction, satisfying transparency requirements and enabling actionable insights beyond a simple risk score.

**Limitation:** AUC 0.68 reflects the fundamental difficulty of injury prediction — many causal factors (training load, match intensity, individual physiology) are unobserved. The model is best used as a risk screening tool alongside clinical judgment, not as a definitive predictor.

## Conclusions

| Feature Category | Key Features | Direction |
|---|---|---|
| Injury history | avg_days_injured_prev_seasons, significant_injury_prev_season | Positive → High Risk |
| Career load | cumulative_days_injured, cumulative_minutes_played | Positive → High Risk |
| Age | age | Positive → older = higher risk |
| Physical | BMI, weight_kg | Marginal positive |
| Position | position_numeric | Variable |
| Work rate | work_rate_numeric | Near zero |

The model learns **medically coherent patterns**: past injury and cumulative workload dominate injury prediction, which aligns with sports medicine literature. This validates the model is capturing real signal, not noise — while also confirming that injury prediction remains inherently uncertain at AUC 0.68.